In [ ]:
%pip install rouge_score bert_score evaluate
from datasets import load_dataset, Dataset
import pandas as pd
import numpy as np
import random as rd
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, Trainer, TrainingArguments, AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
from tqdm import tqdm
import evaluate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from peft import LoraConfig, get_peft_model, TaskType
from evaluate import load
from sklearn.model_selection import train_test_split

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0", device_map="cuda", torch_dtype=torch.bfloat16)

In [ ]:
ds_train = load_dataset("FreedomIntelligence/RAG-Instruct", split="train[:80%]")
ds_test = load_dataset("FreedomIntelligence/RAG-Instruct", split="train[80%:]")
rankings_train = np.load('/kaggle/input/train-document-score/xenc_scores_train-stsb-distilroberta-base.npy')
rankings_test = np.load('/kaggle/input/test-score-npy/xenc_scores_test-stsb-distilroberta-base.npy')

k = 3

In [ ]:
def build_df(ds, doc_rankings):
    docs = [d for s in ds['documents'] for d in s]
    questions = [q for q in ds['question']]
    answers = [a for a in ds['answer']]
    
    data = []

    for i, (q, a) in enumerate(zip(questions, answers)):
        ranked_indices = [int(t[1]) for t in doc_rankings[i][:k]]
        top_docs = [docs[i + idx] for idx in ranked_indices]
        data.append({
            'question': q,
            'answer': a,
            'topk_documents': top_docs
        })

    df = pd.DataFrame(data)
    topk_df = Dataset.from_pandas(df)
    return topk_df

In [ ]:
topk_ds_train = build_df(ds_train, rankings_train)
topk_ds_testval = build_df(ds_test, rankings_test)

topk_ds_split = topk_ds_testval.train_test_split(
    test_size=0.25,
    shuffle=True,
    seed=42
)

topk_ds_val = topk_ds_split["train"]
topk_ds_test = topk_ds_split["test"] 

### Performance Evaluation

In [ ]:
import re

def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)  # Collapse multiple spaces
    return text

def generate_answer(model, tokenizer, question, context, max_length=512):
    prompt = f"### Question:\n{question}\n\n### Context:\n{context}\n\n### Answer:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            num_beams=4,
            early_stopping=True
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("### Answer:")[-1].strip()

from tqdm.notebook import tqdm
import evaluate

def evaluate_model(model, tokenizer, dataset, max_samples=5, max_new_tokens=50, print_examples=True):
    import torch
    import evaluate
    from tqdm import tqdm

    model.eval()

    dataset = dataset.select(range(min(len(dataset), max_samples)))

    bleu = evaluate.load("bleu")
    rouge = evaluate.load("rouge")

    predictions = []
    references = []

    for i, example in enumerate(tqdm(dataset, desc="Evaluating")):
        question = example["question"]
        context = example["topk_documents"]
        reference_text = example["answer"]

        # Puoi personalizzare questo prompt se hai uno stile più adatto
        input_text = f"Domanda: {question}\nContesto: {context}\nRisposta:"

        inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=1
            )

        decoded_output = tokenizer.decode(output_ids[0], skip_special_tokens=True)

        predictions.append(decoded_output)
        references.append(reference_text)

        if print_examples:
            print(f"\n--- Example {i+1} ---")
            print(f"Input:\n{input_text}")
            print(f"Prediction:\n{decoded_output}")
            print(f"Reference:\n{reference_text}")

    bleu_score = bleu.compute(predictions=predictions, references=[[ref] for ref in references])
    rouge_score = rouge.compute(predictions=predictions, references=references)

    return {
        "bleu": bleu_score["bleu"],
        "rougeL": rouge_score["rougeL"]
    }

print(topk_ds_test.column_names)
print(topk_ds_test.shape)

In [ ]:
base_metrics = evaluate_model(
    model=model,
    tokenizer=tokenizer,
    dataset=topk_ds_test,
    max_samples=topk_ds_test.shape[0],
    print_examples=False
)

#Base model: 
# Metrics:
#  bleu: 0.0236
#  rougeL: 0.1061

print("\nMetrics:")
for k, v in base_metrics.items():
    print(f"{k}: {v:.4f}")

### LoRA FT

In [ ]:
from itertools import chain

def preprocess_function(examples, tokenizer):
    inputs = []
    for question, docs, answer in zip(examples["question"], examples["topk_documents"], examples["answer"]):
        context = " ".join(chain.from_iterable(docs)) if isinstance(docs[0], list) else " ".join(docs)
        if isinstance(answer, list):
            answer = " ".join(map(str, answer))
        else:
            answer = str(answer)

        prompt = f"### Question:\n{question}\n\n### Context:\n{context}\n\n### Answer:\n"
        input_text = prompt + answer
        inputs.append(input_text)

    tokenized = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length"
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized


In [ ]:
def create_lora_model(model):
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        inference_mode=False,
        r=8, 
        lora_alpha=32,
        lora_dropout=0.1,
        target_modules=["q_proj", "v_proj"]
    )
    
    return get_peft_model(model, lora_config)

In [ ]:
def pd_to_hf_ds(dataset, tokenizer):
    if isinstance(dataset, pd.DataFrame):
        dataset = Dataset.from_pandas(dataset)
    
    tokenized_dataset = dataset.map(lambda x: preprocess_function(x, tokenizer), batched=True)
    return tokenized_dataset

In [ ]:
model = create_lora_model(model)
model.print_trainable_parameters()

topk_hf_train = pd_to_hf_ds(topk_ds_train, tokenizer)
topk_hf_val = pd_to_hf_ds(topk_ds_val, tokenizer)

training_args = TrainingArguments(
    output_dir="/kaggle/working/train",
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    fp16=True,
    gradient_accumulation_steps=4,
    report_to="tensorboard",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=topk_hf_train,
    eval_dataset=topk_hf_val,
    tokenizer=tokenizer,
)

In [ ]:
trainer.train()

model.save_pretrained("tinyllama-qa-1")
tokenizer.save_pretrained("tinyllama-qa-1")

In [ ]:
ft_metrics = evaluate_model(model, tokenizer, topk_ds_test)

print("\nMetrics:")
for k, v in base_metrics.items():
    print(f"{k}: {v:.4f}")

In [ ]:

def preprocess_function(example):
    # Construct the prompt format for TinyLlama (causal LM)
    prompt = f"### Question:\n{example['question']}\n\n### Context:\n{example['context']}\n\n### Answer:\n"
    input_text = prompt + example["answer"]
    model_inputs = tokenizer(input_text, max_length=512, truncation=True, padding="max_length")
    model_inputs["labels"] = model_inputs["input_ids"].copy()
    return model_inputs


In [ ]:

from transformers import Trainer, TrainingArguments


In [ ]:

import math

def compute_metrics(eval_preds):
    loss = eval_preds.loss
    try:
        perplexity = math.exp(loss)
    except OverflowError:
        perplexity = float("inf")
    return {"perplexity": perplexity}
